# Dashboard and explainability check (phase 3)

Runs on the **finished tuning notebook's output**: no training here.

**Before running:** Settings → **GPU T4** · **Internet ON** · *Add Input* → your dataset (with `manifest.csv`) **and**
*Add Input → Your Work → the finished tuning notebook* (its output contains `tuning/deploy_bundle`).
Optional: a Kaggle secret `ANTHROPIC_API_KEY` so the plain-language summaries are written by Claude.

| Section | What it does | Rough GPU time |
|---|---|---|
| 1 | unit tests of the explanation methods, the summary and the web API | 2 min |
| 2 | one BUS-BRA image through the dashboard API: prediction, Grad-CAM, LIME, SHAP, summary | 1 min |
| 3 | explanation quality on 120 external (BUS-BRA) and 120 internal test images | 15–25 min |
| 4 | downloads | 1 min |

## 0. Setup

In [ ]:
import os, sys, importlib
REPO_URL = "https://github.com/vp-y/mtp.git"
REPO = "/kaggle/working/repo"

# Only if the repo is PRIVATE: add a Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

# step out of the repo first: re-running this cell deletes it, and a deleted working directory breaks git
os.chdir("/kaggle/working")
%cd /kaggle/working
!rm -rf {REPO}
!git clone --depth 1 -b main {REPO_URL} {REPO}
assert os.path.isdir(f"{REPO}/src/bus"), "Clone FAILED - read the git message above"
%cd {REPO}
!pip install -q -r requirements-kaggle.txt

# make the 'bus' package visible to this notebook AND to every !python command
SRC = f"{REPO}/src"
os.environ["PYTHONPATH"] = SRC
sys.path_importer_cache.pop(SRC, None)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
!python -c "import bus; print('bus OK ->', bus.__file__)"

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert hits, "Dataset not found: add the dataset that contains manifest.csv via 'Add Input'"
DATA = os.path.dirname(hits[0])
os.environ["BUS_DATA_ROOT"] = DATA
print("data:", DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, zipfile
X = "/kaggle/working/xai"                      # explainability results go here
os.makedirs(X, exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/deploy_bundle/bundle.json", recursive=True)
               + glob.glob("/kaggle/working/**/deploy_bundle/bundle.json", recursive=True))
if not found:  # maybe only the zip was attached
    zips = glob.glob("/kaggle/input/**/deploy_bundle.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working")
        found = glob.glob("/kaggle/working/deploy_bundle/bundle.json")
assert found, ("No deploy_bundle found: Add Input -> Your Work -> the finished TUNING notebook "
               "(its output contains tuning/deploy_bundle), or upload deploy_bundle.zip as a dataset")
BUNDLE = os.path.dirname(found[0])
print("bundle:", BUNDLE)
!python -m bus.bundle info {BUNDLE}

In [ ]:
# Optional: plain-language summaries written by Claude. Add a Kaggle secret named ANTHROPIC_API_KEY
# (Add-ons -> Secrets, key from console.anthropic.com). Without it the built-in template text is used.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    print("ANTHROPIC_API_KEY loaded: summaries will be written by Claude")
except Exception:
    os.environ.pop("ANTHROPIC_API_KEY", None)
    print("no ANTHROPIC_API_KEY secret: summaries use the built-in template")

## 1. Tests

In [ ]:
!pip install -q "fastapi>=0.100" python-multipart httpx "anthropic>=2.0"
!python -m pytest -q tests/test_explain.py

## 2. One image through the dashboard API
Exactly what the browser does: `/api/predict` (prediction + Grad-CAM), `/api/explain` (LIME, SHAP), `/api/summary` (plain-language text).

In [ ]:
import base64, cv2, numpy as np, pandas as pd
from IPython.display import Markdown, HTML, display
os.environ["BUS_BUNDLE"] = BUNDLE
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from fastapi.testclient import TestClient
from app.server import app

m = pd.read_csv(f"{DATA}/manifest.csv")
row = m[(m.dataset == "BUS-BRA") & (m.label == "malignant")].iloc[0]
img_bytes = open(f"{DATA}/{row.image_path}", "rb").read()

def show_images(*urls, titles=()):
    cells = "".join(f'<figure style="margin:0"><img src="{u}" style="width:260px"><figcaption>{t}</figcaption></figure>'
                    for u, t in zip(urls, titles))
    display(HTML(f'<div style="display:flex;gap:10px;flex-wrap:wrap">{cells}</div>'))

with TestClient(app) as client:
    print(client.get("/api/info").json()["llm"])
    d = client.post("/api/predict", files={"file": ("scan.png", img_bytes, "image/png")}).json()
    print(f"{row.id} (truth: {row.label}) -> {d['decision']} | score {d['prob_malignant']} | threshold {d['threshold']:.3f}")
    xs = {}
    for method in ("lime", "shap"):
        r = client.post(f"/api/explain/{d['case_id']}?method={method}")
        assert r.status_code == 200, r.text
        xs[method] = r.json()
        print(method, "took", xs[method]["elapsed_ms"], "ms; deletion check:", xs[method]["metrics"]["deletion_check"])
    show_images(d["overlay"], d["gradcam"], xs["lime"]["image"], xs["shap"]["image"],
                titles=("outline", "Grad-CAM", "LIME", "SHAP"))
    s = client.post(f"/api/summary/{d['case_id']}", json={"language": "English", "audience": "patient"}).json()
    print("summary source:", s["source"], s.get("model") or "", s.get("note") or "")
    display(Markdown(s["text"]))
    assert client.get("/").status_code == 200
print("dashboard OK")

## 3. How good are the explanations?
Grad-CAM attention inside the **ground-truth** lesion (vs. the chance level `gt_share_of_region`), pointing game, LIME / SHAP deletion test and LIME fit, for correct vs. wrong decisions.

In [ ]:
!python -m bus.explain_eval --bundle {BUNDLE} --scenario A --role external --n 120 --methods gradcam lime shap --out {X}/external

In [ ]:
!python -m bus.explain_eval --bundle {BUNDLE} --scenario deploy --role test --n 120 --methods gradcam lime shap --out {X}/deploy_test

In [ ]:
import json
from IPython.display import Image
for part in ("external", "deploy_test"):
    s = json.load(open(f"{X}/{part}/summary.json"))
    print(f"\n== {part}: {s['n_lesion_found']}/{s['n_images']} images with a lesion found")
    print(pd.DataFrame({k: s[k] for k in ("all", "correct", "wrong") if s.get(k)}).round(3).to_string())
    display(Image(f"{X}/{part}/examples.png"))

## 4. Downloads

In [ ]:
%cd /kaggle/working
!rm -f xai_results.zip
!zip -qr xai_results.zip xai
!ls -lh xai_results.zip
from IPython.display import FileLink
display(FileLink("xai_results.zip"))